# Types of Memory

**Prerequisites:** `02_stores.ipynb` and `03_operations.ipynb` from this chapter.

Three names come up whenever memory is discussed, usually as a list and rarely with anything hanging on them:

- **Semantic** — facts. *The checkout pool is 200 connections.*
- **Episodic** — events. *Checkout latency crossed two seconds on the 14th of October.*
- **Procedural** — how to do things. *How this agent calls a tool; what a good answer looks like.*

Taken as vocabulary they are not worth much. What makes them worth a notebook is that **the same operation is correct for one and destructive for another**, and nothing in the machinery built so far can tell which it is looking at.

The sharpest case is already sitting in `03_operations`. Its write path searches for near-duplicates and decides **add / update / ignore**, which is exactly right for a fact that has changed. Point it at two deploy failures a week apart and it deletes one of them — measured below at **five times out of five**.

**By the end you'll have:**
- Placed each type in a store by asking what will be asked of it, which is the same question notebook 2 used
- Watched a vector index fail to answer *which of these happened last*, with four events 0.0037 apart
- Found that semantic memory in a graph works and buys nothing a dict lookup did not already do
- Seen that most procedural memory is not in a store at all, and which slice of it is
- Destroyed a run history with notebook 3's own write path, then fixed it by telling the write what kind of thing it was recording

## Setup

Everything here opens what notebooks 2 and 3 left behind.

## The three types, in the stores you already have

Nothing new needs building to see them. The corpus from notebook 3 contains all three:

Read them side by side and the difference is not subject matter — all three are about the same handful of services. It is **what makes each one wrong**.

- A **fact** goes wrong by being *superseded*. When the pool is resized again, entry #22 stops being true and something has to replace it.
- An **event** cannot go wrong that way. The 6th of November happened. A later incident does not make it untrue, and nothing should ever replace it.
- A **rule** goes wrong by being *revoked* or *changed* — closer to a fact than an event.

That is the whole distinction, and everything else in this notebook follows from it. **Facts get replaced. Events accumulate.**

### Why every list of memory types looks incomplete

Because there are two axes, and most lists collapse them into one. The three types above say **what a memory is about**. Separately, memory has a **lifetime** — how long it survives — and that is where "working memory", "short-term memory" and "the scratchpad" live.

Crossed, they close:

| | Semantic (facts) | Episodic (events) | Procedural (how-to) |
|---|---|---|---|
| **Working** — this turn | facts pasted into the prompt | the recent messages | the system prompt, the tool schemas |
| **Short-term** — this session | extracted entities | the conversation | the scratchpad, the current plan |
| **Long-term** — across sessions | a user profile | run history | learned skills, few-shot examples |

Notebook 1's `Scratchpad` and its saved conversation are not *additional kinds* of memory. They are two cells in this table — short-term procedural and short-term episodic — which is why they needed no store and why nothing in notebooks 2 and 3 replaced them.

The rest of this notebook is the bottom row, because that is the row with a storage decision in it.

## Episodic memory in a vector index

The natural place to put an event is wherever the prose goes, which means the journal. Here are four deploy failures — the same sentence with one number changed, which is what a run history actually looks like:

**0.0037 between the best and the worst.** They are the same sentence, so of course they are: an embedding encodes what a sentence *means*, and these four mean nearly the same thing. The question asked which was **last**, and "last" is not a property of meaning — it is a property of a column the index never saw.

The November entry does come out on top, by twelve ten-thousandths over the July one. That is luck, not knowledge. The two in the middle are returned out of order entirely.

The same four events, in the store shaped for ordering them:

Exact, ordered, and countable. `ORDER BY at DESC LIMIT 1` is not an approximation of anything, and **`COUNT` has no equivalent in a vector index at any corpus size** — the question *how many times has this happened* cannot be asked of similarity at all.

So an event wants a column with a date in it. That does not banish it from the journal — notebook 3's entries carry an `at` column precisely so they can be filtered and ordered — but it does mean **an event stored only as prose has lost the thing that makes it an event.**

## Semantic memory in a graph

The opposite mistake, and a more tempting one because it works.

A fact is a subject, a relation and an object — *the user's timezone is Asia/Kolkata* — which is exactly an edge. So it can go in the graph, and the graph will answer:

Both correct. One took a Cypher query against a server in another country, the other took a dictionary lookup, and neither is more right than the other.

Which is the point. **A graph earns its place when the answer is in the connections**, as in notebook 2's question 5 — *who else touched this service, and what did they decide* — where no single memory holds the answer and you have to walk. A fact with one value under one name has no connections to walk. Putting it in a graph adds a network round trip, a running server, and a query language, in exchange for the answer a dict already had.

This is the same warning notebook 2 opened with, arriving from the other direction: **later in the list is not better.** A store is chosen by what will be asked, and *"what is this user's timezone"* asks for a lookup no matter how sophisticated the thing answering it is.

## Procedural memory is mostly not in a store

The third type is the one that breaks the pattern, and it is worth being blunt about it rather than inventing a store for symmetry.

Procedural memory is *how to do things*. Most of an agent's is already written down, in two places that have been there since chapter 3:

That description is procedural memory. The model reads it before every decision, and it says two things: here is something you can do, and here is when to do it.

It persists. It changes how the agent behaves. And it lives in a decorator, not a database. The system prompt is the same thing at a larger size.

So most procedural memory needs no store at all.

**The part that does is the part the agent works out for itself.** That this phrasing gets better answers. That this customer's invoices always need the manual path. Nobody wrote those down in advance, so they have nowhere to go unless you give them somewhere.

Where they go follows the usual rule: what will be asked of them. Examples you fetch by name go in a `KeyValue`. Examples you pick by resemblance go in a journal. There is no fourth kind of store here.

### Getting a procedure in front of the model

A tool's description is free. It is already being sent on every call.

Most procedures are not tool descriptions. *How to run a backfill safely.* *Who to page and when.* Those have to reach the prompt some other way, and something has to decide when.

The obvious approach is to store them like any other memory and retrieve the relevant ones.

Three such rules are already in the journal. Here are five situations where one of them should apply:

**Two of the five never arrived.**

Look at the third one. *"A payments retry storm doubled outbound requests, what now?"* That is exactly what the retry rule is for. It scored **0.2717** — nowhere near the top three.

The question reads like an incident report. The journal is full of incident reports. Five of them look more like *"what now?"* than a one-line policy does.

The backfill rule did better. It scored **0.7034**, which is a good score, and it still missed. Three stories about backfills going wrong beat it.

**The rules are not hard to find. They are outnumbered.**

A procedure kept among memories competes with everything the agent has ever experienced. And an agent keeps experiencing things.

### Keep them somewhere else

Stop ranking procedures against experiences. Put them in their own list.

Once they are separate there are only a few of them — and a few things do not need ranking at all.

This shape is called a **skill**. It has two parts:

- **description** — one line saying *when to use this*
- **body** — the actual instructions

The description stays in the prompt permanently. The body is loaded only if the description gets picked.

Four descriptions. 502 characters. In the prompt on every turn.

The bodies are the other 642 characters, and they cost nothing until something asks for them. In a real skill that gap is far wider: a description is one line, a body is a page of instructions and often a folder of scripts.

Now the same five situations, decided by reading that list instead of searching the journal:

**Fifteen picks, fifteen right** — including the one that scored 0.2717 before.

Nothing clever is happening. **There is no ranking.** All four descriptions are in the prompt, so the model picks from a list it can see. Four things cannot push each other out.

And the reason is not that descriptions are better written than rules. It is that there are four of them instead of twenty-four, and none of them are incident reports.

The description still matters. It is what the model reads when choosing, and *"use whenever a payment has failed"* is easier to pick from than a bare instruction.

**This runs out eventually.** Every description is paid for on every turn. Four cost 502 characters. Four hundred would not fit, and ranking them brings the competition straight back.

That is fine, because **procedures do not pile up the way memories do.** The journal grows every time the agent learns anything. The catalogue grows only when someone decides the agent should be able to do a new thing.

### The part of a rule that does need a database

One thing a system prompt is bad at: **saying what it used to be.**

A rule changes — the batch cap moves from five hundred rows to two hundred — and with prompts and code the old version survives only in whatever version control the repository happens to have. For a rule an agent revised *itself*, at three in the morning, there is no commit.

This is the one place procedural memory wants a table, and it wants it for the reason tables have always been good: an append-only history you can read backwards.

Three rows, not one row edited three times — which is the same choice `Records` made in `03_operations` by having no `update()` at all. *When did we decide five hundred was too many, and what was it before?* is answerable here and unanswerable in a prompt.

The graph is the other candidate the moment rules start depending on each other — *this rule only applies while that tool is enabled* — and it is usually more machinery than a handful of rules deserves. Worth knowing the option exists before reaching for it.

## So which store for which type?

There is an obvious mapping, and it is worth writing down precisely because two thirds of it is right:

> episodic → relational · semantic → vectors · procedural → the prompt

**Episodic → relational** is correct, and already measured above: events need ordering and counting, and 0.0037 of cosine spread cannot provide either.

**Procedural → the prompt** is correct, with the table above for the part that needs a history.

**Semantic → vectors is the trap**, and it is the one this chapter opened by warning about. Watch the same *kind* of memory — a fact about the world — behave completely differently depending on whether you can name it:

Both of the first two are semantic memory. One takes **four microseconds** and is exact; the other takes nearly two seconds, most of it an embedding call over the network, and returns a ranking.

The third block is the part that matters. There is no timezone anywhere in that journal, and the vector index answers anyway — *"Releases go out on Tuesdays"*, scored `0.5835`, which looks like a number you could trust. `03_operations` built a floor for exactly this, and a floor is a patch on a question that should never have been asked of this store.

So the type does not pick the store. **Nameability does**, which is the same criterion notebook 2 used all along — what will be asked of it:

| Type | The question it tends to get | Store |
|---|---|---|
| Episodic | *when*, *how many*, *which was last* | relational |
| Semantic, nameable | *what is X* | key–value |
| Semantic, not nameable | *what do I know about X* | vectors |
| Procedural | *how do I do X* | the prompt and the code; a table when it needs a history |

Read the middle two rows together: **semantic memory is not one storage decision.** "The user's timezone is Asia/Kolkata" and "checkout's pool was resized because the per-node share had fallen too low" are both facts, and they belong in different stores, because only one of them has a name you could ask for.

The type is a useful prior — it tells you which questions are *likely* — and it is not the decision. Which is why this notebook's real payoff is not the table above, but the branch it puts in the write path.

## Where the types actually diverge: the write path

Everything above is about *where* a memory goes. This is about what happens when it arrives, and it is the reason the three names are worth knowing at all.

`03_operations` built a write path that searches before it writes and then decides **add / update / ignore**. It exists so the same fact does not end up in the journal five times, and for facts it works — measured there at three correct decisions out of three.

Here is that same procedure, unchanged, looking at two deploy failures nine days apart. The dates are in the `at` column, where dates live, so the text of the two entries differs by one word:

**Five out of five.** Every run decided that the 12th of August *updates* the 3rd of August, which in `03_operations`' write path means `journal.update(31, ...)` — the earlier deploy's text overwritten, the row reused, the failure on the 3rd gone with no error raised anywhere.

And the procedure is not malfunctioning. Asked *"does this new text state the same thing as one of these with a detail that replaces it?"*, the honest answer about two near-identical sentences is yes. The question was wrong, not the answer.

Notice also what made it certain rather than occasional. With the dates written into the prose — *"failed on 3 August"* against *"failed on 12 August"* — the same check gets it right about two times in three. The dates are not in the prose, because they are in a column, which is the correct place for them. **The one field that distinguishes the two events is the one field the duplicate check never sees.**

So the fix is not a better prompt for the same question. It is asking a different question, which requires knowing what kind of memory this is:

Nothing clever happened there. Two changes: the date each memory is about went into the listing, and the prompt said *this is an event, and two events that read alike are still two events.*

Which is the chapter's own lesson again, from `03_operations`' two-stage recall: **whatever the decision depends on has to be in front of the thing deciding.** It was true of a model choosing which memory to open, and it is true of a model deciding whether a memory is a duplicate.

### Routing the write by type

Once the type is known, most of the decision disappears. An event never needs a duplicate check at all — it is a new occurrence by definition, and searching before writing it is a model call spent to be told `add`:

An event costs one insert and no model call. A fact costs a search and a decision, because it might be replacing something.

That is the practical return on the three names. Not a taxonomy to recite — a branch in the write path, where taking the wrong arm quietly deletes history that nothing will ever tell you is missing.

## What you built

✓ Separated the three types by **what makes each one wrong**, rather than by subject: a fact is superseded, an event cannot be, a rule is revoked — so **facts get replaced and events accumulate**

✓ Closed the usual list of memory types by crossing it with a second axis, lifetime, and found that `Scratchpad` and the saved conversation were never additional kinds — they are two cells in the table, which is why they needed no store

✓ Watched a vector index fail to say which of four events happened last, with **0.0037** between best and worst, and the middle two returned out of order — because "last" is a property of a column, not of meaning

✓ Answered the same question with `ORDER BY at DESC LIMIT 1`, and noted the one question similarity cannot be asked at all: **`COUNT`**

✓ Put a fact in a graph, watched it work, and left it out anyway — a graph earns its place when the answer is in the connections, and a value under a name has none to walk

✓ Found most procedural memory already written down in tool descriptions and the system prompt, with only the *learned* slice needing a store — plus the one thing a prompt cannot do, which is say what it used to be, so a changed rule wants an append-only table rather than an edit

✓ Found the reason a stored procedure goes missing: it is not hard to find, it is **outnumbered**. Three policies among twenty-four memories lost two of five situations to incident reports phrased like the questions — and that competition worsens every time the agent learns anything

✓ Fixed it by moving procedures out rather than ranking them better — a **skill catalogue**: one-line descriptions always in context, bodies loaded only when chosen. Fifteen selections, fifteen correct, with no ranking involved at all

✓ Named the reason it works, which is not the flattering one: not better wording, just four candidates instead of twenty-four

✓ Rejected the obvious type-to-store mapping where it is wrong: **semantic memory is not one storage decision.** A nameable fact is four microseconds and exact; an unnameable one is two seconds and ranked; and the nameable one asked of a vector index returns *"Releases go out on Tuesdays"* at `0.5835` from a journal containing no timezone at all

✓ Destroyed a run history with `03_operations`' own write path: **five times out of five**, a deploy failure on the 12th was recorded as an update to the one on the 3rd, silently

✓ Traced why it was certain rather than occasional — the dates live in a column, so the one field distinguishing the two events is the one field the duplicate check never sees

✓ Fixed it at **five out of five** by putting the dates in the listing and saying what kind of memory it was, which is the same lesson two-stage recall taught: whatever the decision depends on has to be in front of the thing deciding

✓ Routed the write by type, so an event costs one insert and no model call, and only a fact pays for a search

**Next:** `05_memory_in_the_loop.ipynb` — wiring all of it back into `02_agent_runtime`, which turns out to need no modification: a component is anything with `.run(conversation)`, so recall slots in beside `Observe`. Recall by injection before `Think`, write by tool — and the one thing genuinely missing, which is anywhere on `Conversation` to hang a store.